In [ ]:
# 최초 1회 실행: 필요한 패키지 설치
%pip install -U ortools networkx shapely geopandas pandas numpy pyproj pyogrio google-genai streamlit pydeck


In [ ]:
import math
import networkx as nx

from shapely.geometry import LineString, MultiLineString
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from pyproj import Transformer


# ============================================================
# 기본 설정
# ============================================================

# ============================================================
# 배포용 경로 설정
#
# 권장 구조:
# project/
# ├─ app.py
# ├─ generate_demo_data.ipynb
# └─ data/
#    ├─ seoul.shp
#    ├─ seoul.shx
#    ├─ seoul.dbf
#    ├─ seoul.prj
#    └─ ...
#
# 현재 노트북이 실행되는 프로젝트 폴더를 기준으로 data/ 사용
# ============================================================

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ROAD_SHP_PATH = DATA_DIR / "seoul.shp"

if not ROAD_SHP_PATH.exists():
    raise FileNotFoundError(
        f"도로망 파일을 찾을 수 없습니다: {ROAD_SHP_PATH}\n"
        "data 폴더에 seoul.shp와 관련 SHP 구성파일을 넣어주세요."
    )

print("프로젝트 폴더:", PROJECT_DIR)
print("데이터 폴더:", DATA_DIR)


# ============================================================
# 1. 서울 도로망 읽기
# ============================================================

roads = gpd.read_file(ROAD_SHP_PATH)

print("도로 수:", len(roads))
print("원본 CRS:", roads.crs)

# 앱과 동일하게 EPSG:5179로 변환
roads_m = roads.to_crs(epsg=5179)

print("변환 CRS:", roads_m.crs)
print("도로 범위:", roads_m.total_bounds)


# ============================================================
# 2. 실제 도로망 중심좌표
# ============================================================

xmin, ymin, xmax, ymax = roads_m.total_bounds

CENTER_X = (xmin + xmax) / 2
CENTER_Y = (ymin + ymax) / 2

print("중심 X:", CENTER_X)
print("중심 Y:", CENTER_Y)


# ============================================================
# 3. 좌표 변환기
# ============================================================

transformer = Transformer.from_crs(
    "EPSG:5179",
    "EPSG:4326",
    always_xy=True
)


def to_lonlat(x, y):
    return transformer.transform(x, y)


# ============================================================
# 4. 물류센터
# ============================================================

lon, lat = to_lonlat(CENTER_X, CENTER_Y)

depot_df = pd.DataFrame([
    {
        "depot_id": 1,
        "name": "물류센터",
        "node_x": CENTER_X,
        "node_y": CENTER_Y,
        "lon": lon,
        "lat": lat,
    }
])


# ============================================================
# 5. 차량 5대
# ============================================================

driver_offsets = [
    (-1200, -700),
    (900, -1100),
    (-1500, 1200),
    (1300, 900),
    (400, 1600),
]

drivers = []

for vehicle_id, (dx, dy) in enumerate(
    driver_offsets,
    start=1
):

    x = CENTER_X + dx
    y = CENTER_Y + dy

    lon, lat = to_lonlat(x, y)

    drivers.append({
        "driver_id": vehicle_id,
        "vehicle_id": vehicle_id,
        "node_x": x,
        "node_y": y,
        "lon": lon,
        "lat": lat,
        "available": 1,
        "capacity": 15,
    })

drivers_df = pd.DataFrame(drivers)


# ============================================================
# 6. 배송지 40개
# ============================================================

rng = np.random.default_rng(42)

NUM_DELIVERIES = 40

delivery_offsets = []

while len(delivery_offsets) < NUM_DELIVERIES:

    dx = int(
        rng.integers(
            -3000,
            3001
        )
    )

    dy = int(
        rng.integers(
            -2500,
            2501
        )
    )

    # 물류센터와 너무 가까운 배송지는 제외
    distance = np.sqrt(
        dx ** 2
        + dy ** 2
    )

    if distance < 500:
        continue

    delivery_offsets.append(
        (
            dx,
            dy
        )
    )


# ------------------------------------------------------------
# ★ 중요: 실제 deliveries_df 생성
# ------------------------------------------------------------

deliveries = []

for delivery_id, (dx, dy) in enumerate(
    delivery_offsets,
    start=1
):

    x = CENTER_X + dx
    y = CENTER_Y + dy

    lon, lat = to_lonlat(x, y)

    # 차량 총 적재용량이 75이므로
    # 각 배송지 물량을 1~2로 설정
    demand = int(
        rng.integers(
            1,
            3
        )
    )

    deliveries.append({
        "delivery_id": delivery_id,
        "name": f"배송지_{delivery_id:02d}",
        "node_x": x,
        "node_y": y,
        "lon": lon,
        "lat": lat,
        "demand": demand,
    })


deliveries_df = pd.DataFrame(
    deliveries
)


# ------------------------------------------------------------
# 배송지 생성 확인
# ------------------------------------------------------------

print("\n===== 배송지 생성 확인 =====")
print("목표 배송지 수:", NUM_DELIVERIES)
print("실제 배송지 수:", len(deliveries_df))

print(
    "배송지 이름:",
    deliveries_df["name"].tolist()
)

print(
    "총 배송물량:",
    int(deliveries_df["demand"].sum())
)

print(
    "총 차량용량:",
    int(drivers_df["capacity"].sum())
)

if len(deliveries_df) != NUM_DELIVERIES:
    raise ValueError(
        f"배송지 생성 오류: "
        f"{len(deliveries_df)}개 생성됨 "
        f"(목표 {NUM_DELIVERIES}개)"
    )

if deliveries_df["name"].nunique() != NUM_DELIVERIES:
    raise ValueError(
        "배송지 이름이 중복되었습니다."
    )

if (
    deliveries_df["demand"].sum()
    > drivers_df["capacity"].sum()
):
    raise ValueError(
        "총 배송물량이 차량 총 적재용량을 초과했습니다."
    )


# ============================================================
# 7. 기존 현장지식
# ============================================================

knowledge_df = pd.DataFrame([
    {
        "knowledge_id": "K001",
        "location": "배송지_03",
        "knowledge_type": "congestion",
        "description": "오후 시간대 주변 도로가 혼잡함",
        "extra_delay_min": 15,
    },
    {
        "knowledge_id": "K002",
        "location": "배송지_07",
        "knowledge_type": "unloading",
        "description": "하역 시간이 일반 배송지보다 오래 걸림",
        "extra_delay_min": 20,
    },
    {
        "knowledge_id": "K003",
        "location": "배송지_12",
        "knowledge_type": "access",
        "description": "대형 차량 진입이 어려움",
        "extra_delay_min": 10,
    },
])


# ============================================================
# 8. 하루 전체 초기 배송경로 최적화
# ============================================================
# - 실제 SHP 도로망 이동시간 사용
# - 현장지식 추가 지연시간 반영
# - 차량 5대 전체 배차 및 방문순서 동시 결정
# ============================================================

print("\n초기 배송경로 최적화 시작")


# ============================================================
# 8-1. 배송지/차량 주변 도로만 사용
# ============================================================

all_x = pd.concat([
    depot_df["node_x"],
    drivers_df["node_x"],
    deliveries_df["node_x"]
])

all_y = pd.concat([
    depot_df["node_y"],
    drivers_df["node_y"],
    deliveries_df["node_y"]
])

MARGIN_M = 5000

xmin_area = all_x.min() - MARGIN_M
xmax_area = all_x.max() + MARGIN_M

ymin_area = all_y.min() - MARGIN_M
ymax_area = all_y.max() + MARGIN_M

roads_local = roads_m.cx[
    xmin_area:xmax_area,
    ymin_area:ymax_area
].copy()

print(
    "최적화 사용 도로 수:",
    len(roads_local)
)

if roads_local.empty:
    raise ValueError(
        "배송지역 주변 도로가 없습니다."
    )


# ============================================================
# 8-2. SHP → NetworkX 도로 그래프
# ============================================================

G = nx.DiGraph()

DEFAULT_SPEED_KMH = 30.0

speed_col = None

for col in [
    "MAX_SPD",
    "MAX_SPEED",
    "SPEED",
    "SPD"
]:

    if col in roads_local.columns:
        speed_col = col
        break


def round_node(x, y):

    return (
        round(float(x), 2),
        round(float(y), 2)
    )


def add_line_to_graph(
    G,
    line,
    speed_kmh
):

    coords = list(line.coords)

    if len(coords) < 2:
        return

    speed_mps = (
        speed_kmh
        * 1000
        / 3600
    )

    for i in range(
        len(coords) - 1
    ):

        u = round_node(
            coords[i][0],
            coords[i][1]
        )

        v = round_node(
            coords[i + 1][0],
            coords[i + 1][1]
        )

        if u == v:
            continue

        length_m = math.hypot(
            v[0] - u[0],
            v[1] - u[1]
        )

        if length_m <= 0:
            continue

        travel_time_sec = (
            length_m
            / speed_mps
        )

        # MVP에서는 양방향 도로로 처리
        G.add_edge(
            u,
            v,
            length_m=length_m,
            travel_time_sec=travel_time_sec
        )

        G.add_edge(
            v,
            u,
            length_m=length_m,
            travel_time_sec=travel_time_sec
        )


for _, row in roads_local.iterrows():

    speed_kmh = DEFAULT_SPEED_KMH

    if speed_col is not None:

        try:

            value = float(
                row[speed_col]
            )

            if (
                np.isfinite(value)
                and value > 0
            ):
                speed_kmh = value

        except Exception:
            pass

    geom = row.geometry

    if isinstance(
        geom,
        LineString
    ):

        add_line_to_graph(
            G,
            geom,
            speed_kmh
        )

    elif isinstance(
        geom,
        MultiLineString
    ):

        for line in geom.geoms:

            add_line_to_graph(
                G,
                line,
                speed_kmh
            )


print(
    "도로 그래프:",
    G.number_of_nodes(),
    "nodes /",
    G.number_of_edges(),
    "edges"
)

if G.number_of_nodes() == 0:
    raise ValueError(
        "도로 그래프 생성에 실패했습니다."
    )


# ============================================================
# 8-3. 가장 큰 연결 네트워크 사용
# ============================================================

components = list(
    nx.weakly_connected_components(G)
)

largest_component = max(
    components,
    key=len
)

G = G.subgraph(
    largest_component
).copy()

graph_nodes = np.array(
    list(G.nodes),
    dtype=float
)


def nearest_graph_node(
    x,
    y
):

    d2 = (
        (graph_nodes[:, 0] - float(x)) ** 2
        + (graph_nodes[:, 1] - float(y)) ** 2
    )

    idx = np.argmin(d2)

    return tuple(
        graph_nodes[idx]
    )


# ============================================================
# 8-4. 물류센터 / 차량 / 배송지를 도로망에 연결
# ============================================================

depot_node = nearest_graph_node(
    depot_df.iloc[0]["node_x"],
    depot_df.iloc[0]["node_y"]
)


driver_nodes = {}

for _, row in drivers_df.iterrows():

    vehicle_id = int(
        row["vehicle_id"]
    )

    driver_nodes[vehicle_id] = (
        nearest_graph_node(
            row["node_x"],
            row["node_y"]
        )
    )


delivery_nodes = {}

for _, row in deliveries_df.iterrows():

    delivery_nodes[row["name"]] = (
        nearest_graph_node(
            row["node_x"],
            row["node_y"]
        )
    )


# ============================================================
# 8-5. 현장정보 → 배송지 서비스시간
# ============================================================

DEFAULT_SERVICE_MIN = 5

extra_delay = {
    name: 0
    for name in deliveries_df["name"]
}

for _, row in knowledge_df.iterrows():

    location = row["location"]

    if location in extra_delay:

        extra_delay[location] += int(
            row["extra_delay_min"]
        )


service_time_min = {
    name:
        DEFAULT_SERVICE_MIN
        + extra_delay[name]

    for name in deliveries_df["name"]
}


print("\n배송지 서비스시간")

for name in service_time_min:

    if extra_delay[name] > 0:

        print(
            name,
            ":",
            service_time_min[name],
            "분"
        )


# ============================================================
# 8-6. OR-Tools용 노드 구성
# ============================================================
#
# 차량 5대
# 배송지 40개
# 물류센터 1개
#
# 총 location node:
# 0 ~ 4   : 차량 출발지
# 5 ~ 44  : 배송지 40개
# 45      : 물류센터
#
# 총 46개
# ============================================================

vehicle_ids = (
    drivers_df["vehicle_id"]
    .astype(int)
    .tolist()
)

delivery_names = (
    deliveries_df["name"]
    .tolist()
)

num_vehicles = len(
    vehicle_ids
)

print("\n===== OR-Tools 노드 확인 =====")
print("차량 수:", num_vehicles)
print("배송지 수:", len(delivery_names))
print("예상 전체 노드 수:", 5 + len(delivery_names) + 1)


location_nodes = []
location_types = []
location_names = []


# ------------------------------------------------------------
# 차량 시작 위치
# ------------------------------------------------------------

vehicle_start_indices = []

for vehicle_id in vehicle_ids:

    vehicle_start_indices.append(
        len(location_nodes)
    )

    location_nodes.append(
        driver_nodes[vehicle_id]
    )

    location_types.append(
        "vehicle_start"
    )

    location_names.append(
        f"차량_{vehicle_id}_출발"
    )


# ------------------------------------------------------------
# 배송지
# ------------------------------------------------------------

delivery_index_map = {}

for delivery_name in delivery_names:

    idx = len(location_nodes)

    delivery_index_map[
        delivery_name
    ] = idx

    location_nodes.append(
        delivery_nodes[
            delivery_name
        ]
    )

    location_types.append(
        "delivery"
    )

    location_names.append(
        delivery_name
    )


# ------------------------------------------------------------
# 물류센터
# ------------------------------------------------------------

depot_index = len(
    location_nodes
)

location_nodes.append(
    depot_node
)

location_types.append(
    "depot"
)

location_names.append(
    "물류센터"
)


# 모든 차량의 종료점 = 물류센터
vehicle_end_indices = [
    depot_index
] * num_vehicles


print(
    "실제 location node 수:",
    len(location_nodes)
)

if len(location_nodes) != (
    num_vehicles
    + NUM_DELIVERIES
    + 1
):

    raise ValueError(
        "OR-Tools 노드 수가 예상과 다릅니다."
    )


# ============================================================
# 8-7. 실제 도로망 기준 이동시간 행렬
# ============================================================

n_locations = len(
    location_nodes
)

travel_time_matrix = np.zeros(
    (
        n_locations,
        n_locations
    ),
    dtype=int
)

travel_distance_matrix = np.zeros(
    (
        n_locations,
        n_locations
    ),
    dtype=float
)

path_cache = {}


def shortest_metrics(
    start_node,
    end_node
):

    key = (
        start_node,
        end_node
    )

    if key in path_cache:
        return path_cache[key]

    try:

        path = nx.shortest_path(
            G,
            start_node,
            end_node,
            weight="travel_time_sec"
        )

    except nx.NetworkXNoPath:

        return (
            10**9,
            10**9
        )

    total_time_sec = 0.0
    total_distance_m = 0.0

    for i in range(
        len(path) - 1
    ):

        edge = G[
            path[i]
        ][
            path[i + 1]
        ]

        total_time_sec += float(
            edge["travel_time_sec"]
        )

        total_distance_m += float(
            edge["length_m"]
        )

    result = (
        total_time_sec,
        total_distance_m
    )

    path_cache[key] = result

    return result


print(
    "\n도로 이동시간 행렬 계산 중..."
)

for i in range(
    n_locations
):

    for j in range(
        n_locations
    ):

        if i == j:
            continue

        time_sec, distance_m = (
            shortest_metrics(
                location_nodes[i],
                location_nodes[j]
            )
        )

        travel_time_matrix[
            i,
            j
        ] = int(
            round(time_sec)
        )

        travel_distance_matrix[
            i,
            j
        ] = distance_m


print(
    "이동시간 행렬 계산 완료"
)


# ============================================================
# 8-8. OR-Tools VRP 모델
# ============================================================

manager = pywrapcp.RoutingIndexManager(
    n_locations,
    num_vehicles,
    vehicle_start_indices,
    vehicle_end_indices
)

routing = pywrapcp.RoutingModel(
    manager
)


# ============================================================
# 8-9. 이동시간 + 배송 서비스시간
# ============================================================

def time_callback(
    from_index,
    to_index
):

    from_node = (
        manager.IndexToNode(
            from_index
        )
    )

    to_node = (
        manager.IndexToNode(
            to_index
        )
    )

    travel_sec = int(
        travel_time_matrix[
            from_node,
            to_node
        ]
    )

    service_sec = 0

    name = location_names[
        from_node
    ]

    if (
        location_types[
            from_node
        ]
        == "delivery"
    ):

        service_sec = int(
            service_time_min[name]
            * 60
        )

    return (
        travel_sec
        + service_sec
    )


transit_callback_index = (
    routing.RegisterTransitCallback(
        time_callback
    )
)

routing.SetArcCostEvaluatorOfAllVehicles(
    transit_callback_index
)


# ============================================================
# 8-10. 배송 물량 + 차량 적재용량 제약
# ============================================================

delivery_demand = {
    row["name"]:
        int(row["demand"])

    for _, row in deliveries_df.iterrows()
}

vehicle_capacities = (
    drivers_df
    .sort_values("vehicle_id")
    ["capacity"]
    .astype(int)
    .tolist()
)


def demand_callback(
    from_index
):

    node = manager.IndexToNode(
        from_index
    )

    if (
        location_types[node]
        != "delivery"
    ):
        return 0

    delivery_name = (
        location_names[node]
    )

    return int(
        delivery_demand[
            delivery_name
        ]
    )


demand_callback_index = (
    routing.RegisterUnaryTransitCallback(
        demand_callback
    )
)

routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,
    vehicle_capacities,
    True,
    "Capacity"
)


# ============================================================
# 8-11. 차량별 최대 하루 운행시간
# ============================================================

MAX_ROUTE_MIN = 360

routing.AddDimension(
    transit_callback_index,
    0,
    MAX_ROUTE_MIN * 60,
    True,
    "Time"
)

time_dimension = (
    routing.GetDimensionOrDie(
        "Time"
    )
)


# 차량 간 운행시간 편차도 줄이도록 설정
time_dimension.SetGlobalSpanCostCoefficient(
    20
)


# ============================================================
# 8-12. 최적화 설정
# ============================================================

search_parameters = (
    pywrapcp.DefaultRoutingSearchParameters()
)

search_parameters.first_solution_strategy = (
    routing_enums_pb2
    .FirstSolutionStrategy
    .PATH_CHEAPEST_ARC
)

search_parameters.local_search_metaheuristic = (
    routing_enums_pb2
    .LocalSearchMetaheuristic
    .GUIDED_LOCAL_SEARCH
)

search_parameters.time_limit.seconds = 20


# ============================================================
# 8-13. 최적화 실행
# ============================================================

solution = routing.SolveWithParameters(
    search_parameters
)

if solution is None:

    raise ValueError(
        "초기 배송경로 최적화에 실패했습니다."
    )


# ============================================================
# 8-14. 최적 차량별 배송순서 추출
# ============================================================

routes = {}

route_summary = []

visited_deliveries = set()


for vehicle_idx, vehicle_id in enumerate(
    vehicle_ids
):

    index = routing.Start(
        vehicle_idx
    )

    delivery_route = []

    total_distance_m = 0.0

    while not routing.IsEnd(
        index
    ):

        from_node = (
            manager.IndexToNode(
                index
            )
        )

        next_index = (
            solution.Value(
                routing.NextVar(
                    index
                )
            )
        )

        to_node = (
            manager.IndexToNode(
                next_index
            )
        )

        if (
            location_types[
                from_node
            ]
            == "delivery"
        ):

            delivery_name = (
                location_names[
                    from_node
                ]
            )

            delivery_route.append(
                delivery_name
            )

            visited_deliveries.add(
                delivery_name
            )

        total_distance_m += (
            travel_distance_matrix[
                from_node,
                to_node
            ]
        )

        index = next_index


    routes[
        vehicle_id
    ] = delivery_route


    end_time_sec = (
        solution.Value(
            time_dimension.CumulVar(
                routing.End(
                    vehicle_idx
                )
            )
        )
    )


    total_demand = sum(
        delivery_demand[name]
        for name in delivery_route
    )


    route_summary.append({

        "vehicle_id":
            vehicle_id,

        "deliveries":
            len(delivery_route),

        "total_demand":
            total_demand,

        "capacity":
            int(
                drivers_df.loc[
                    drivers_df["vehicle_id"]
                    == vehicle_id,
                    "capacity"
                ].iloc[0]
            ),

        "route":
            (
                " → ".join(
                    delivery_route
                )
                if delivery_route
                else "-"
            ),

        "distance_km":
            (
                total_distance_m
                / 1000
            ),

        "estimated_time_min":
            (
                end_time_sec
                / 60
            )
    })


route_summary_df = pd.DataFrame(
    route_summary
)


# ============================================================
# 배송지 누락 여부 확인
# ============================================================

missing_deliveries = sorted(
    set(delivery_names)
    - visited_deliveries
)

duplicate_count = (
    len(visited_deliveries)
)

print(
    "\n===== 배송지 배차 확인 ====="
)

print(
    "생성된 배송지:",
    len(delivery_names)
)

print(
    "배차된 배송지:",
    duplicate_count
)

print(
    "미배차 배송지:",
    len(missing_deliveries)
)

if missing_deliveries:

    print(
        "미배차 목록:",
        missing_deliveries
    )

else:

    print(
        "모든 배송지가 배차되었습니다."
    )


print(
    "\n===== 초기 최적 배차 결과 ====="
)

display(
    route_summary_df
)


# ============================================================
# 8-15. app.py용 routes_before_v2.csv 생성
# ============================================================

route_rows = []

for vehicle_id in vehicle_ids:

    delivery_list = routes[
        vehicle_id
    ]

    if delivery_list:

        route_text = (
            f"차량 {vehicle_id} 현재위치 → "
            + " → ".join(
                delivery_list
            )
            + " → 물류센터"
        )

    else:

        route_text = (
            f"차량 {vehicle_id} 현재위치 "
            f"→ 물류센터"
        )

    route_rows.append({

        "vehicle_id":
            vehicle_id,

        "route":
            route_text
    })


routes_before_df = pd.DataFrame(
    route_rows
)


print(
    "\n최종 routes_before_v2.csv"
)

display(
    routes_before_df
)


# ============================================================
# 9. CSV 저장
# ============================================================

depot_df.to_csv(
    DATA_DIR / "depot.csv",
    index=False,
    encoding="utf-8-sig"
)

drivers_df.to_csv(
    DATA_DIR / "drivers.csv",
    index=False,
    encoding="utf-8-sig"
)

deliveries_df.to_csv(
    DATA_DIR / "deliveries.csv",
    index=False,
    encoding="utf-8-sig"
)

knowledge_df.to_csv(
    DATA_DIR / "knowledge.csv",
    index=False,
    encoding="utf-8-sig"
)

routes_before_df.to_csv(
    DATA_DIR / "routes_before_v2.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 10. 최종 확인
# ============================================================

print("\n========================================")
print("데이터 재생성 완료")
print("========================================")

print(
    "물류센터:",
    CENTER_X,
    CENTER_Y
)

print(
    "배송지 수:",
    len(deliveries_df)
)

print(
    "차량 수:",
    len(drivers_df)
)

print(
    "총 배송물량:",
    int(
        deliveries_df["demand"].sum()
    )
)

print(
    "총 차량용량:",
    int(
        drivers_df["capacity"].sum()
    )
)

print(
    "배송지 CSV:",
    DATA_DIR / "deliveries.csv"
)

print(
    "경로 CSV:",
    DATA_DIR / "routes_before_v2.csv"
)

print("========================================")

## Gemini API 키 설정

API 키는 노트북이나 GitHub 저장소에 직접 저장하지 않습니다.\
로컬/개인 실행 시 아래 셀에서 환경변수로 입력하거나, 배포 환경의 Secrets 기능을 사용하세요.


In [ ]:
import os
import getpass

# 이미 환경변수에 등록되어 있으면 그대로 사용
# 없으면 실행할 때만 입력받음(노트북 파일에 저장되지 않음)
if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Gemini API Key: ")

print("GEMINI_API_KEY 설정 완료")


In [ ]:
import subprocess
import sys
from pathlib import Path

APP_PATH = Path.cwd() / "app.py"

if not APP_PATH.exists():
    raise FileNotFoundError(f"app.py를 찾을 수 없습니다: {APP_PATH}")

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(APP_PATH),
    "--server.port",
    "8501",
])

print("Streamlit 실행 요청 완료")
print("로컬 실행: http://localhost:8501")
print("GitHub Codespaces에서는 아래 '포트' 탭의 8501 전달 주소를 여세요.")
